In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
from plot_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:

def get_both_dfs(fname):
    #print(fname)
    df1 = pd.read_csv(fname).drop(columns='Unnamed: 0').set_index('sample')
    df2 = pd.read_csv(fname.split('parent1_info.csv')[0] + 'parent2_info.csv').drop(columns='Unnamed: 0').set_index('sample')
    df2['conf_int'] = df2['boot_high']-df2['boot_low']
    
    df1['conf_int'] = df1['boot_high']-df1['boot_low']
    df_both = pd.concat([df1.rename(columns = {'boot_med':'boot_med1', 'boot_low':'boot_low1', 'boot_high':'boot_high1',
       'actual_med':'actual_med1', 'conf_int':'conf_int1'}),
                         df2.rename(columns = {'boot_med':'boot_med2', 'boot_low':'boot_low2', 'boot_high':'boot_high2',
       'actual_med':'actual_med2', 'conf_int':'conf_int2'})],axis=1)
    
    df_both['species'] = fname.split('/')[-2]
    df_both['fname'] = fname
   # print('-'.join(fname1.split('/')[-1].split('-')[:-1]))
   # parent_media = 
    
    df_both['subjects_measured'] = '-'.join(fname.split('/')[-1].split('-')[:-1])
    df_both['in_measured'] = '-'.join(fname.split('/')[-1].split('_')[:-2])
    df_both['total_shift'] = np.abs(1-(df_both['actual_med1'] + df_both['actual_med2']))
    df_both['total_shift12'] = np.abs(1-(df_both['boot_low1'] + df_both['boot_high2']))
    df_both['total_shift21'] = np.abs(1-(df_both['boot_low2'] + df_both['boot_high1']))
    df_both['total_shift_max'] = df_both['total_shift21']
    df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift_max'] = df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift12']

    return df_both

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4.csv').set_index('sample')

for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        fname1=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if os.path.exists(fname1):
            
            df_both=get_both_dfs(fname1)
            df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
            df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
       # except:
        #    continue
            df_both_meta['species_id']=sp
            df_both_meta['diff_from_in1']=df_both_meta['actual_med1']
            df_both_meta['diff_from_in2']=df_both_meta['actual_med2']
            in_samples = df_both_meta['inoculumn_sample'].unique()
            if len(in_samples) != len(df_both_meta.loc[df_both_meta['sample'].isin(in_samples),'actual_med1']):
                continue
            
            for in_sample in df_both_meta['inoculumn_sample'].unique():
                
                with_in_sample = df_both_meta.loc[df_both_meta['inoculumn_sample']==in_sample,'sample'].values
                in1_val=df_both_meta.loc[df_both_meta['sample']==in_sample,'actual_med1'].values[0]
                df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in1']= \
                    df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in1'] - in1_val
                
                in2_val=df_both_meta.loc[df_both_meta['sample']==in_sample,'actual_med2'].values[0]
                df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in2']= \
                    df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in2'] - in2_val
            all_dfs.append(df_both_meta)
all_dfs=pd.concat(all_dfs)
all_dfs['species-type_meso']=all_dfs['species_id'].astype(str)+'-'+all_dfs['type_mesocosm']
all_dfs['species-meso']=all_dfs['species_id'].astype(str)+'-'+all_dfs['mesocosm']
all_dfs = all_dfs.loc[all_dfs['total_shift']<.1,:]
all_dfs['actual_med1']=all_dfs['actual_med1']/(all_dfs['actual_med1']+all_dfs['actual_med2'])
#all_dfs_gr=all_dfs.groupby(['species-type_meso']).median(numeric_only=True).reset_index()

In [ ]:
extinctions_low = {}
extinctions_high = {}
total_extinctions = []
thresh = 1e-3
for passage in [7,6,5,4,3,2,1,0]:
    passage_df = all_dfs.loc[all_dfs['passage']==passage,:]
    extincts_low = passage_df.loc[(passage_df['actual_med1']<=thresh),'species-meso'].unique()
    extincts_high = passage_df.loc[(passage_df['actual_med1']>=1-thresh),'species-meso'].unique()
    if passage < 7:
        extincts_low = np.intersect1d(extinctions_low[passage+1],extincts_low)
        extincts_high = np.intersect1d(extinctions_high[passage+1],extincts_high)
    extinctions_low[passage]=extincts_low
    extinctions_high[passage]=extincts_high
    total_extinctions.append(len(extincts_low)+len(extincts_high))
passage_df = all_dfs.loc[all_dfs['passage']==7,:]
alive= passage_df.loc[(passage_df['actual_med1']>thresh)*(passage_df['actual_med1']<1-thresh),'species-meso'].unique()
total_extinctions.append(len(alive))
    

In [ ]:
iqplot.ecdf

In [ ]:
b = passage_df.loc[(passage_df['actual_med1']>1e-3)*(passage_df['actual_med1']<1-1e-3),:].copy()
b.loc[b['actual_med1']>.5,'actual_med1' ]=1-b.loc[b['actual_med1']>.5,'actual_med1']
b['actual_med1']=np.log10(b['actual_med1'])
p = iqplot.ecdf(b, q = 'actual_med1', )
bokeh.io.show(p)

In [ ]:
#passage_df.loc[(passage_df['actual_med1']>=1e-3)*(passage_df['actual_med1']<=1-1e-3),'species-meso']

In [ ]:
passages = np.array([7,6,5,4,3,2,1,0,'Both Alive']).astype(str)
data = pd.DataFrame(data={'passage':passages,'n_extinctions': total_extinctions })
p = hv.Scatter(data.sort_values(by='passage'), vdims = ['passage', 'n_extinctions'],kdims=['passage','n_extinctions']).opts(
    color=bokeh.palettes.Bright[3][1],size=10,width=400, xlabel='Passage', ylabel='Cumulative Extinctions')
p2 = hv.Scatter(data.loc[data['passage']=='Both Alive',:], vdims = ['passage', 'n_extinctions'],kdims=['passage','n_extinctions']).opts(
    color=bokeh.palettes.Bright[5][0],size=10,width=400, xlabel='Passage', ylabel='Cumulative Extinctions',ylim=(0,250))
p=hv.render(p*p2)
p.yaxis.axis_label_text_font_style = 'normal'
p.xaxis.axis_label_text_font_style = 'normal'
p.output_backend='svg'
export_plot_pdf(p,'num_extinctions_time')

In [ ]:
passages = np.array([7,6,5,4,3,2,1,0,'still_alive']).astype(str)
data = pd.DataFrame(data={'passage':passages,'n_extinctions': total_extinctions })
p = hv.Scatter(data.sort_values(by='passage'), vdims = ['passage', 'n_extinctions'],kdims=['passage','n_extinctions']).opts(color=bokeh.palettes.Bright[3][1],
                                                                                                  size=10)
p